# IE 7615 — Milestone 2: YOLOv8 Object Detection

## Objective

This notebook builds and evaluates a custom YOLOv8 object detection pipeline for the Milestone 2 multi-object dataset containing 73 object classes.

The workflow generates multi-object composite images from the original single-object dataset, creates YOLO-format annotations automatically, trains multiple YOLOv8 model variants, selects the best model using validation performance, and evaluates the selected model on a held-out test set.

### Pipeline

1. Verify the Google Colab GPU environment
2. Clone and synchronize the project repository
3. Audit the original source dataset
4. Create leakage-safe train, validation, and test source splits
5. Generate multi-object composite images and YOLO annotations
6. Validate the generated detection dataset
7. Install Ultralytics
8. Train YOLOv8 Nano
9. Train YOLOv8 Small
10. Compare validation performance and select the final model
11. Evaluate the selected model on the held-out test set
12. Run inference and return Object IDs with bounding-box locations
13. Save final prediction figures

### Final Dataset

- 73 object classes
- 400 multi-object composite images
- 3–5 objects per image
- 1,606 bounding-box annotations
- Train: 280 images / 1,122 annotations
- Validation: 60 images / 247 annotations
- Test: 60 images / 237 annotations
- Random seed: 42

## 1. Verify GPU Environment

YOLO training is performed using a GPU in Google Colab.

This step verifies:
- PyTorch version
- CUDA availability
- assigned GPU model

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available.")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


## 2. Clone and Synchronize the Project Repository

The repository contains:
- original single-object dataset
- Object ID mapping
- dataset auditing scripts
- leakage-safe source splitting
- multi-object dataset generation
- YOLO dataset configuration
- dataset validation scripts

The repository is synchronized with the latest `main` branch before rebuilding the dataset.

The generated detection dataset is not stored in GitHub because it can be reproduced from the source images and project scripts.

In [2]:
# Clone the latest version of the project repository.
!git clone https://github.com/pjraghuram/IE7615-Deep-Learning-for-AI.git

# Move into the repository.
%cd IE7615-Deep-Learning-for-AI

Cloning into 'IE7615-Deep-Learning-for-AI'...
remote: Enumerating objects: 7470, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 7470 (delta 17), reused 54 (delta 15), pack-reused 7409 (from 1)
Receiving objects: 100% (7470/7470), 168.62 MiB | 18.22 MiB/s, done.
Resolving deltas: 100% (48/48), done.
Updating files: 100% (7386/7386), done.
/content/IE7615-Deep-Learning-for-AI


In [3]:
# Confirm Colab is using the latest main branch from GitHub.
!git fetch origin
!git reset --hard origin/main

print("\nLatest commit:")
!git log -1 --oneline

print("\nActive branch:")
!git branch

print("\nFinal generator check:")
!grep -n "MIN_OBJECTS_PER_COMPOSITE" milestone2/src/generate_final_dataset.py

print("\nValidator check:")
!grep -n "EXPECTED_TOTAL_BOXES" milestone2/src/validate_yolo_dataset.py

HEAD is now at eb43bbf Merge pull request #3 from pjraghuram/milestone2-final

Latest commit:
eb43bbf (HEAD -> main, origin/main, origin/HEAD) Merge pull request #3 from pjraghuram/milestone2-final

Active branch:
* main

Final generator check:
74:MIN_OBJECTS_PER_COMPOSITE = 3
836:        MIN_OBJECTS_PER_COMPOSITE
842:            f"{MIN_OBJECTS_PER_COMPOSITE}-"
1279:            MIN_OBJECTS_PER_COMPOSITE
1726:                        MIN_OBJECTS_PER_COMPOSITE,
1949:        MIN_OBJECTS_PER_COMPOSITE,

Validator check:
25:EXPECTED_TOTAL_BOXES = 1606
582:    if total_boxes != EXPECTED_TOTAL_BOXES:
585:            f"{EXPECTED_TOTAL_BOXES} "


## 3. Audit the Original Source Dataset

Before generating the detection dataset, the original single-object dataset is audited.

The audit verifies:
- 73 object classes are present
- Object IDs match `OBJ001` through `OBJ073`
- image formats are recognized
- background-only images are identified
- exact duplicates are detected
- Object IDs and source folders are consistent
- `images_OBJ054/OBJ054_017.jpg` is explicitly excluded from the usable source dataset

No source images are modified during this step.

In [4]:
!python milestone2/src/audit_source_dataset.py


SOURCE DATASET AUDIT
Class folders:           73
CSV class mappings:      73
Included image files:    7342
Explicitly excluded:     1
Object images:           6957
Background-only images:  385

Explicit exclusions:
  images_OBJ054/OBJ054_017.jpg
  Exclusion validation: PASS

Extensions:
  .jpeg: 207
  .jpg: 7135

Per-class total image range:
  Minimum: 99
  Maximum: 113

Per-class object-image range:
  Minimum: 90
  Maximum: 106

Background-image range:
  Minimum: 0
  Maximum: 11

Exact duplicate check:
  Duplicate hash groups: 28
  Extra duplicate files: 28

Filename checks:
  All filenames contain a recognizable OBJ ID.
  Folder/filename class IDs agree.

Class ID validation: PASS
Explicit exclusion validation: PASS

Manifest saved to:
  milestone2/metadata/source_manifest.csv

Audit complete.


## 4. Create Leakage-Safe Source Splits

The usable single-object source images are split into training, validation, and test sets before any multi-object composites are generated.

This prevents source-image leakage across the final dataset splits.

The split process:
- removes exact duplicate source images from reuse
- keeps train, validation, and test source pools separate
- prevents duplicate hashes from crossing dataset splits
- preserves all 73 object classes across the three splits
- uses a fixed random seed for reproducibility

In [5]:
!python milestone2/src/create_source_splits.py


SOURCE SPLIT CREATION
All source images:        7342
Object-image candidates:  6957
Background images ignored: 385
Unique object images:     6941
Duplicate object files:   16

Split counts:
  Train: 4822
  Val:   1019
  Test:  1100
  Total: 6941

Unique object images per class:
  Minimum: 89
  Maximum: 106

Leakage checks:
  Source path overlap: PASS
  Exact duplicate overlap: PASS
  All 73 classes in train: PASS
  All 73 classes in val:   PASS
  All 73 classes in test:  PASS

Configuration:
  Ratios: 0.70 / 0.15 / 0.15
  Random seed: 42

Files created:
  milestone2/metadata/source_split_manifest.csv
  milestone2/metadata/duplicate_report.csv

Source split creation complete.


## 5. Generate the Final Multi-Object Dataset

The final detection dataset is generated from the leakage-safe source splits.

For each composite image:
- 3 to 5 unique object classes are selected
- each source image is resized/cropped into a 224×224 tile
- tiles are placed in compact, non-overlapping layouts
- YOLO bounding-box labels are generated automatically for every pasted source-image region
- each bounding box corresponds to the exact 224×224 region placed in the composite and may include some background around the object
- YOLO class IDs range from 0 to 72 and map to the canonical Object IDs `OBJ001` through `OBJ073`
- train, validation, and test composites are created only from their corresponding source split

### Composite Layouts

- 3 objects → 2×2 layout with one empty cell, 448×448 canvas
- 4 objects → 2×2 layout, 448×448 canvas
- 5 objects → 2×3 layout with one empty cell, 672×448 canvas

The final generated dataset contains 400 composite images with 1,606 total annotations.

In [6]:
!python milestone2/src/generate_final_dataset.py


FINAL MILESTONE 2 DATASET GENERATION

Generating images for: Hemanth Rajaboyina

Generating images for: Jaya Raghu Ram Penugonda

Generating images for: Nikhil Kudupudi

Generating images for: Sri Lakshmi Swetha Jalluri

Dataset summary:
  Train composites: 280
  Val composites:   60
  Test composites:  60
  Total composites: 400
  Total annotations: 1606

Objects per composite:
  3 objects: 132
  4 objects: 130
  5 objects: 138

Compact layouts:
  3 objects -> 2x2 grid / 448x448 canvas: 132
  4 objects -> 2x2 grid / 448x448 canvas: 130
  5 objects -> 2x3 grid / 672x448 canvas: 138

Per-member allocation:
  Hemanth Rajaboyina: 100
  Jaya Raghu Ram Penugonda: 100
  Nikhil Kudupudi: 100
  Sri Lakshmi Swetha Jalluri: 100

Class-balance check:
  train: min=15, max=16
  val: min=3, max=4
  test: min=3, max=4

Integrity checks:
  Explicit exclusion absent: PASS
  No source reuse: PASS
  No MD5 reuse: PASS
  No cross-split source leakage: PASS
  No cross-split MD5 leakage: PASS
  3-5 objects

## 6. Validate the Generated YOLO Dataset

The generated dataset is validated before model training.

The validation checks:
- every image has a matching YOLO label file
- each label follows normalized YOLO format
- class IDs are within the valid range 0–72
- bounding-box coordinates remain inside image boundaries
- each composite contains 3–5 labeled objects
- object classes are unique within each composite
- all 73 classes are represented
- train, validation, and test split counts are correct
- the final dataset contains 1,606 total bounding-box annotations

Training proceeds only after all validation checks pass.

In [7]:
!python milestone2/src/validate_yolo_dataset.py


YOLO DATASET VALIDATION
Dataset root: /content/IE7615-Deep-Learning-for-AI/milestone2/data/generated
Classes:      73

TRAIN
  Images:       280
  Labels:       280
  Boxes:        1122
  3-object imgs: 93
  4-object imgs: 92
  5-object imgs: 95
  Class range:  15–16
  Pairing:      PASS
  Label format: PASS
  Coordinates:  PASS
  Unique class: PASS
  Class cover:  PASS

VAL
  Images:       60
  Labels:       60
  Boxes:        247
  3-object imgs: 16
  4-object imgs: 21
  5-object imgs: 23
  Class range:  3–4
  Pairing:      PASS
  Label format: PASS
  Coordinates:  PASS
  Unique class: PASS
  Class cover:  PASS

TEST
  Images:       60
  Labels:       60
  Boxes:        237
  3-object imgs: 23
  4-object imgs: 17
  5-object imgs: 20
  Class range:  3–4
  Pairing:      PASS
  Label format: PASS
  Coordinates:  PASS
  Unique class: PASS
  Class cover:  PASS

GLOBAL SUMMARY
  Images:         400
  Label files:    400
  Bounding boxes: 1606
  Classes:        73

Objects per composite:
 

## 7. Install Ultralytics

Ultralytics provides the YOLOv8 implementation used for Milestone 2.

The package is installed only after the dataset passes all validation checks. The package version is pinned for reproducibility.

In [8]:
!pip install -q ultralytics==8.4.174

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 9.4 MB/s eta 0:00:00


In [9]:
import torch
import ultralytics

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.174
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


## 8. Train YOLOv8 Nano

The pretrained `yolov8n.pt` model is fine-tuned on the final 73-class multi-object dataset.

### Training Configuration

- Epochs: 50
- Early-stopping patience: 10
- Input image size: 640
- Batch size: 16
- Random seed: 42
- Deterministic training enabled
- Pretrained YOLOv8 weights used for initialization

The validation split is used during training and later for model comparison. The held-out test split is not used for model selection.

In [10]:
from ultralytics import YOLO

model_n = YOLO("yolov8n.pt")

results_n = model_n.train(
    data="milestone2/configs/data.yaml",
    epochs=50,
    patience=10,
    imgsz=640,
    batch=16,
    seed=42,
    deterministic=True,
    project="milestone2/results/runs",
    name="yolov8n_final",
)

New https://pypi.org/project/ultralytics/8.4.175 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=milestone2/configs/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=

## 9. Train YOLOv8 Small

The pretrained `yolov8s.pt` model is fine-tuned on the same final dataset using the same training configuration as YOLOv8 Nano.

Keeping the training settings consistent allows the two model variants to be compared fairly using validation performance.

### Training Configuration

- Epochs: 50
- Early-stopping patience: 10
- Input image size: 640
- Batch size: 16
- Random seed: 42
- Deterministic training enabled
- Pretrained YOLOv8 weights used for initialization

In [11]:
from ultralytics import YOLO

model_s = YOLO("yolov8s.pt")

results_s = model_s.train(
    data="milestone2/configs/data.yaml",
    epochs=50,
    patience=10,
    imgsz=640,
    batch=16,
    seed=42,
    deterministic=True,
    project="milestone2/results/runs",
    name="yolov8s_final",
)

New https://pypi.org/project/ultralytics/8.4.175 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=milestone2/configs/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=

## 10. Compare Models and Select the Final Model

YOLOv8 Nano and YOLOv8 Small are compared using validation performance.

### Validation Results

| Model | Precision | Recall | mAP@0.5 | mAP@0.5:0.95 |
|---|---:|---:|---:|---:|
| YOLOv8 Nano | 0.662 | 0.533 | 0.632 | 0.614 |
| YOLOv8 Small | **0.890** | **0.836** | **0.897** | **0.896** |

YOLOv8 Small is selected as the final model because it substantially outperforms YOLOv8 Nano across all reported validation metrics.

The held-out test set is not used in this model-selection decision.

## 11. Evaluate the Selected Model on the Held-Out Test Set

After selecting YOLOv8 Small using validation performance, the final model is evaluated once on the held-out test split.

The test set contains:
- 60 composite images
- 237 labeled objects
- all 73 object classes

The test set is used only for final performance reporting and is not used for model selection or additional tuning.

In [12]:
from ultralytics import YOLO

best_model_path = (
    "/content/IE7615-Deep-Learning-for-AI/"
    "runs/detect/milestone2/results/runs/"
    "yolov8s_final/weights/best.pt"
)

final_model = YOLO(best_model_path)

test_results = final_model.val(
    data="milestone2/configs/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
)

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,153,835 parameters, 0 gradients, 28.6 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 11.9±4.1 MB/s, size: 78.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/IE7615-Deep-Learning-for-AI/milestone2/data/generated/labels/test... 60 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 60/60 414.2it/s 0.1s
val: New cache created: /content/IE7615-Deep-Learning-for-AI/milestone2/data/generated/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.2it/s 3.3s
                   all         60        237      0.833      0.801      0.861      0.861
            Basketball          4          4          1      0.928      0.995      0.995
          Water bottle 

### Final Test Results

| Metric | Result |
|---|---:|
| Precision | 0.833 |
| Recall | 0.801 |
| mAP@0.5 | 0.861 |
| mAP@0.5:0.95 | 0.861 |

The selected YOLOv8 Small model maintains strong performance on the held-out test set, with only a moderate decrease from validation performance.

## 12. Run Inference on Held-Out Multi-Object Images

The selected YOLOv8 Small model is applied to five held-out test composites.

A confidence threshold of 0.25 is used for prediction.

The goal is to demonstrate that the trained detector can:
- identify multiple objects in the same image
- return the corresponding Object IDs
- provide confidence scores
- localize each detected object with a bounding box

In [13]:
from ultralytics import YOLO
from pathlib import Path

best_model_path = (
    "/content/IE7615-Deep-Learning-for-AI/"
    "runs/detect/milestone2/results/runs/"
    "yolov8s_final/weights/best.pt"
)

final_model = YOLO(best_model_path)

test_dir = Path("milestone2/data/generated/images/test")

sample_images = sorted(test_dir.glob("*.jpg"))[:5]

predictions = final_model.predict(
    source=[str(p) for p in sample_images],
    imgsz=640,
    conf=0.25,
    save=True
)


0: 640x640 1 Table Tennis Racket, 1 Knife, 1 Sloth Plushie, 1 Bowl, 19.6ms
1: 640x640 1 Rubiks cube, 1 UNO card box, 1 Pencil case, 1 Measuring Bottle, 19.6ms
2: 640x640 1 Candle, 1 Clutcher, 1 Wayfarer Sunglasses, 19.6ms
3: 640x640 1 Toy Bunny, 1 Moisturizer, 1 Men's Wallet, 19.6ms
4: 640x640 1 Mug, 1 Perfume bottle, 1 Tissue, 1 Headphones, 1 Analog watch, 19.6ms
Speed: 1.6ms preprocess, 19.6ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/IE7615-Deep-Learning-for-AI/runs/detect/predict


### Object IDs and Bounding-Box Locations

YOLO internally uses class indices from 0 to 72. These indices are converted back to the required canonical Object ID format `OBJ001` through `OBJ073`.

For each detection, the output includes:
- Object ID
- object name
- confidence score
- bounding-box coordinates `(x1, y1, x2, y2)`

In [14]:
class_names = final_model.names

for image_path, result in zip(sample_images, predictions):
    print(f"\nImage: {image_path.name}")

    for box in result.boxes:
        class_id = int(box.cls[0].item())
        confidence = float(box.conf[0].item())

        object_id = f"OBJ{class_id + 1:03d}"

        x1, y1, x2, y2 = box.xyxy[0].tolist()

        print(
            f"{object_id} | "
            f"{class_names[class_id]} | "
            f"confidence={confidence:.3f} | "
            f"bbox=({x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f})"
        )


Image: test_000001.jpg
OBJ071 | Sloth Plushie | confidence=0.996 | bbox=(223.8, 0.2, 447.9, 223.8)
OBJ069 | Knife | confidence=0.991 | bbox=(0.0, 0.0, 223.6, 223.4)
OBJ073 | Bowl | confidence=0.974 | bbox=(0.0, 223.9, 223.6, 448.0)
OBJ034 | Table Tennis Racket | confidence=0.939 | bbox=(224.1, 223.7, 447.9, 448.0)

Image: test_000002.jpg
OBJ018 | Rubiks cube | confidence=0.995 | bbox=(224.3, 223.9, 447.8, 448.0)
OBJ036 | Pencil case | confidence=0.994 | bbox=(0.0, 223.6, 224.1, 447.9)
OBJ031 | UNO card box | confidence=0.991 | bbox=(224.6, 0.2, 448.0, 223.5)
OBJ063 | Measuring Bottle | confidence=0.923 | bbox=(0.1, 0.8, 223.6, 224.0)

Image: test_000003.jpg
OBJ037 | Candle | confidence=0.988 | bbox=(0.0, 223.7, 223.5, 448.0)
OBJ040 | Clutcher | confidence=0.907 | bbox=(223.6, 0.0, 447.9, 224.2)
OBJ045 | Wayfarer Sunglasses | confidence=0.827 | bbox=(0.3, 0.2, 223.3, 224.5)

Image: test_000004.jpg
OBJ067 | Men's Wallet | confidence=0.994 | bbox=(224.9, 224.5, 448.0, 447.7)
OBJ021 | Toy

## 13. Save Final Prediction Figures

The annotated prediction images generated from the held-out test examples are copied into the final results directory for use in the report and qualitative analysis.

Only figures produced using the corrected final dataset and selected YOLOv8 Small model are retained.

In [15]:
from pathlib import Path
import shutil

report_dir = Path("milestone2/results/final/report_figures")
report_dir.mkdir(parents=True, exist_ok=True)

predict_dir = Path(
    "/content/IE7615-Deep-Learning-for-AI/runs/detect/predict"
)

saved_images = sorted(predict_dir.glob("*.jpg"))

for img_path in saved_images[:5]:
    dest = report_dir / img_path.name
    shutil.copy2(img_path, dest)
    print(f"Saved: {dest}")

Saved: milestone2/results/final/report_figures/test_000001.jpg
Saved: milestone2/results/final/report_figures/test_000002.jpg
Saved: milestone2/results/final/report_figures/test_000003.jpg
Saved: milestone2/results/final/report_figures/test_000004.jpg
Saved: milestone2/results/final/report_figures/test_000005.jpg


In [16]:
old_files = [
    "test_000007_comparison.jpg",
    "test_000008_comparison.jpg",
    "test_000013_comparison.jpg",
    "test_000059_comparison.jpg",
]

for name in old_files:
    path = report_dir / name
    if path.exists():
        path.unlink()
        print(f"Removed old figure: {name}")

Removed old figure: test_000007_comparison.jpg
Removed old figure: test_000008_comparison.jpg
Removed old figure: test_000013_comparison.jpg
Removed old figure: test_000059_comparison.jpg


In [17]:
!ls -lh milestone2/results/final/report_figures

total 420K
-rw-r--r-- 1 root root  84K Oct 10 02:03 test_000001.jpg
-rw-r--r-- 1 root root  90K Oct 10 02:03 test_000002.jpg
-rw-r--r-- 1 root root  59K Oct 10 02:03 test_000003.jpg
-rw-r--r-- 1 root root  80K Oct 10 02:03 test_000004.jpg
-rw-r--r-- 1 root root 101K Oct 10 02:03 test_000005.jpg


## 14. Save Final Training and Evaluation Artifacts

The key outputs from the final YOLOv8 Nano and YOLOv8 Small runs are copied into a single results directory for reporting and GitHub submission.

Saved artifacts include:
- training metrics (`results.csv`)
- training curves (`results.png`)
- confusion matrices
- precision, recall, F1, and PR curves

These files are preserved from the corrected final dataset and final model runs only.

In [18]:
from pathlib import Path
import shutil

final_results_dir = Path("milestone2/results/final")
final_results_dir.mkdir(parents=True, exist_ok=True)

run_dirs = {
    "yolov8n": Path(
        "/content/IE7615-Deep-Learning-for-AI/"
        "runs/detect/milestone2/results/runs/yolov8n_final"
    ),
    "yolov8s": Path(
        "/content/IE7615-Deep-Learning-for-AI/"
        "runs/detect/milestone2/results/runs/yolov8s_final"
    ),
}

files_to_copy = [
    "results.csv",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "BoxF1_curve.png",
    "BoxPR_curve.png",
]

for model_name, run_dir in run_dirs.items():
    model_out = final_results_dir / model_name
    model_out.mkdir(parents=True, exist_ok=True)

    for filename in files_to_copy:
        src = run_dir / filename
        if src.exists():
            dst = model_out / filename
            shutil.copy2(src, dst)
            print(f"Copied: {model_name}/{filename}")
        else:
            print(f"Missing: {model_name}/{filename}")

Copied: yolov8n/results.csv
Copied: yolov8n/results.png
Copied: yolov8n/confusion_matrix.png
Copied: yolov8n/confusion_matrix_normalized.png
Copied: yolov8n/BoxP_curve.png
Copied: yolov8n/BoxR_curve.png
Copied: yolov8n/BoxF1_curve.png
Copied: yolov8n/BoxPR_curve.png
Copied: yolov8s/results.csv
Copied: yolov8s/results.png
Copied: yolov8s/confusion_matrix.png
Copied: yolov8s/confusion_matrix_normalized.png
Copied: yolov8s/BoxP_curve.png
Copied: yolov8s/BoxR_curve.png
Copied: yolov8s/BoxF1_curve.png
Copied: yolov8s/BoxPR_curve.png


In [19]:
!find milestone2/results/final -maxdepth 2 -type f | sort

milestone2/results/final/BoxF1_curve.png
milestone2/results/final/BoxP_curve.png
milestone2/results/final/BoxPR_curve.png
milestone2/results/final/BoxR_curve.png
milestone2/results/final/confusion_matrix_normalized.png
milestone2/results/final/confusion_matrix.png
milestone2/results/final/report_figures/test_000001.jpg
milestone2/results/final/report_figures/test_000002.jpg
milestone2/results/final/report_figures/test_000003.jpg
milestone2/results/final/report_figures/test_000004.jpg
milestone2/results/final/report_figures/test_000005.jpg
milestone2/results/final/results.csv
milestone2/results/final/results.png
milestone2/results/final/yolov8n/BoxF1_curve.png
milestone2/results/final/yolov8n/BoxP_curve.png
milestone2/results/final/yolov8n/BoxPR_curve.png
milestone2/results/final/yolov8n/BoxR_curve.png
milestone2/results/final/yolov8n/confusion_matrix_normalized.png
milestone2/results/final/yolov8n/confusion_matrix.png
milestone2/results/final/yolov8n/results.csv
milestone2/results/fina

In [20]:
from pathlib import Path

final_results_dir = Path("milestone2/results/final")

old_root_files = [
    "BoxF1_curve.png",
    "BoxP_curve.png",
    "BoxPR_curve.png",
    "BoxR_curve.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "results.csv",
    "results.png",
]

for name in old_root_files:
    path = final_results_dir / name
    if path.exists():
        path.unlink()
        print(f"Removed old root-level artifact: {name}")

Removed old root-level artifact: BoxF1_curve.png
Removed old root-level artifact: BoxP_curve.png
Removed old root-level artifact: BoxPR_curve.png
Removed old root-level artifact: BoxR_curve.png
Removed old root-level artifact: confusion_matrix.png
Removed old root-level artifact: confusion_matrix_normalized.png
Removed old root-level artifact: results.csv
Removed old root-level artifact: results.png


In [21]:
!find milestone2/results/final -maxdepth 2 -type f | sort

milestone2/results/final/report_figures/test_000001.jpg
milestone2/results/final/report_figures/test_000002.jpg
milestone2/results/final/report_figures/test_000003.jpg
milestone2/results/final/report_figures/test_000004.jpg
milestone2/results/final/report_figures/test_000005.jpg
milestone2/results/final/yolov8n/BoxF1_curve.png
milestone2/results/final/yolov8n/BoxP_curve.png
milestone2/results/final/yolov8n/BoxPR_curve.png
milestone2/results/final/yolov8n/BoxR_curve.png
milestone2/results/final/yolov8n/confusion_matrix_normalized.png
milestone2/results/final/yolov8n/confusion_matrix.png
milestone2/results/final/yolov8n/results.csv
milestone2/results/final/yolov8n/results.png
milestone2/results/final/yolov8s/BoxF1_curve.png
milestone2/results/final/yolov8s/BoxP_curve.png
milestone2/results/final/yolov8s/BoxPR_curve.png
milestone2/results/final/yolov8s/BoxR_curve.png
milestone2/results/final/yolov8s/confusion_matrix_normalized.png
milestone2/results/final/yolov8s/confusion_matrix.png
mile

In [24]:
# Compress the final Milestone 2 result artifacts into a ZIP file for download.
!zip -r milestone2_final_artifacts.zip milestone2/results/final

updating: milestone2/results/final/ (stored 0%)
updating: milestone2/results/final/report_figures/ (stored 0%)
updating: milestone2/results/final/report_figures/test_000005.jpg (deflated 3%)
updating: milestone2/results/final/report_figures/test_000003.jpg (deflated 4%)
updating: milestone2/results/final/report_figures/test_000002.jpg (deflated 1%)
updating: milestone2/results/final/report_figures/test_000004.jpg (deflated 1%)
updating: milestone2/results/final/report_figures/test_000001.jpg (deflated 1%)
updating: milestone2/results/final/yolov8s/ (stored 0%)
updating: milestone2/results/final/yolov8s/results.csv (deflated 61%)
updating: milestone2/results/final/yolov8s/BoxPR_curve.png (deflated 23%)
updating: milestone2/results/final/yolov8s/BoxP_curve.png (deflated 10%)
updating: milestone2/results/final/yolov8s/BoxR_curve.png (deflated 12%)
updating: milestone2/results/final/yolov8s/results.png (deflated 8%)
updating: milestone2/results/final/yolov8s/confusion_matrix.png (deflated 

In [25]:
from google.colab import files

files.download("milestone2_final_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>